2 layers of mlp with Relu
FFN(X) = max(0, xW1 + b1)W2 + b2

In [ ]:
#though, through multi attention, now the token can noticed each other, but they haven't still 
#thought the info from each other, thah's why we use ffwd to let them think the info from each other
""" 
数据在 ffwd 里的逐层变化

  输入 x            (4, 8, 32)
    │ Linear(32, 128)   ← 每个 token 的 32 维 → 128 维(升维)
    ▼
                   (4, 8, 128)
    │ ReLU()            ← 逐元素非线性,负数变 0,形状不变
    ▼
                   (4, 8, 128)
    │ Linear(128, 32)   ← 128 维 → 32 维(降回原维度)
    ▼
  输出             (4, 8, 32)   ← 和输入同形!
关键点 1:ffwd 是"逐 token 独立"作用的

  这是最重要的理解。nn.Linear 只作用在**最后一维(C=32)**上:

  - 它把 (4, 8, 32) 看成 4×8 = 32 个独立的 token 向量,每个 32 维。
  - 每个 token 向量单独过一遍 32→128→ReLU→32。
  - token 之间互不通信 —— 位置 0 的加工不影响位置 1。

  32 个 token 向量,每个独立地:
  [32维] → Linear → [128维] → ReLU → Linear → [32维]

  用同一套权重(Linear 的参数),但分别作用在每个 token 上。这叫 "position-wise
  feed-forward"(逐位置前馈)。 
  """

In [5]:
import torch
from torch import nn
from torch.nn import functional as F

#hyperparameters
batch_size = 32
block_size = 8
max_iter = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iter = 200
n_embd = 32
# ---------------------

torch.manual_seed(1337)

with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
chars = sorted(list(set(text)))
vocab_size = len(chars)

#map functions
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s] #take a string and map to list of ints
decode = lambda l: "".join([itos[i] for i in l]) #take a list of ints and map to string

#split 
data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9 * len(data))
train = data[:n]
val   = data[n:]

#data load 
def get_batch(split:str):
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out 

class Head(nn.Module):
    "one head of self-attention"
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #tril is not a parameter, put in buffer
    
    def forward(self, x):
        B, T, C = x.shape 
        k = self.key(x)     #BT C = head_size
        q = self.query(x)   #BT C = head_size
        
        #compute attention "affinities"
        wei = q @ k.transpose(-2, -1)  * C**-0.5  #BTT
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
                
        v = self.value(x)   #BT C = head_size
        out = wei @ v #BTT @  #BT C ->  #BT C = head_size
        
        return out 

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        
    def forward(self, x):
        # x BT C = n_embd -> h(x) BT C = head_size
        return torch.cat([h(x) for h in self.heads], dim=-1)        




In [9]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),            
            nn.Linear(4 * n_embd, n_embd)
        )
    
    def forward(self, x):
        return self.net(x)
        

In [10]:
        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        self.sa_heads = MultiHeadAttention(4, n_embd//4) # 4 heads of 8-dim self attention
        self.ffwd = FeedForward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur                
        x = self.sa_heads(x) #The thinking part from the previous tocken, not just self 
        
        #though, through multi attention, now the token can noticed each other, but they haven't still 
        #thought the info from each other, thah's why we use ffwd to let them think the info from each other
        x = self.ffwd(x) #BTC
        
    
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx
    

In [11]:
model = BigramLanguageModel()    
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    

step 0: train loss 4.2067, val loss 4.2052
step 300: train loss 2.6709, val loss 2.6814
step 600: train loss 2.5041, val loss 2.4976
step 900: train loss 2.4307, val loss 2.4196
step 1200: train loss 2.3682, val loss 2.3685
step 1500: train loss 2.3392, val loss 2.3324
step 1800: train loss 2.2947, val loss 2.2972
step 2100: train loss 2.2633, val loss 2.2815
step 2400: train loss 2.2547, val loss 2.2780
step 2700: train loss 2.2475, val loss 2.2548
step 3000: train loss 2.2211, val loss 2.2368
step 3300: train loss 2.1990, val loss 2.2278
step 3600: train loss 2.1862, val loss 2.2091
step 3900: train loss 2.1714, val loss 2.2234
step 4200: train loss 2.1544, val loss 2.1839
step 4500: train loss 2.1485, val loss 2.1891
step 4800: train loss 2.1364, val loss 2.1825


In [16]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


Wher you in emy, ouson joodderqordiney, in bucho if for Youreectertart:
Came faingeentr the clow use of betheajurth, you nid sake Hate, nevees! ay, guled faistsird stat theale uspentheron, thees, cold bewerd I? then, it toleen!
Bonempcare this be Mise shat coustseoxt.

Mord ther ind: Crocge, sher derd his an spue of of pugh setlot if her;
And, be wito llaitpistpows lon tly ap, ond if 'and wottle't lard wous off Catake and that ther And I faithe kone cold Reterts towm.

DUKENVORINCE:
Whor ton!

L


In [ ]:
"""   ┌─────────────────────┬───────────────────────────────────┬─────────────────┐
  │                     │              干什么               │   token 之间    │
  ├─────────────────────┼───────────────────────────────────┼─────────────────┤
  │ attention(sa_heads) │ 让每个 token 收集其他 token       │ 有通信(wei @ v  │
  │                     │ 的信息("交流")                    │ 跨位置聚合)     │
  ├─────────────────────┼───────────────────────────────────┼─────────────────┤
  │ ffwd                │ 让每个 token                      │ 无通信(逐 token │
  │                     │ 独立消化刚收集到的信息("思考")    │  独立)          │
  └─────────────────────┴───────────────────────────────────┴─────────────────┘

  直觉:
  - attention 阶段,位置 4 的 "it" 从位置 1 的 "cat"
  那里拿到了信息(现在它的向量里混了 cat 的成分)。
  - 但"拿到"不等于"想明白"。ffwd 阶段,"it" 这个向量自己过一个小神经网络,把"我是 it
  + 我关注到了 cat"这堆信息加工成更有用的表示。
  - 你注释里 "noticed each other but haven't thought the info" ——
  正是这个意思:attention 让它们注意到彼此,ffwd 让每个 token 消化注意到的东西。 """